## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-16-13-06-06 +0000,nypost,Bank VP fatally struck by bullet shot in fight...,https://nypost.com/2026/11/16/us-news/massachu...
1,2026-11-16-12-57-00 +0000,wsj,"Bitcoin, Ether Slump Further",https://www.wsj.com/finance/currencies/crypto-...
2,2026-11-16-12-56-00 +0000,wsj,The Morning Risk Report: Ground Stop Puts Focu...,https://www.wsj.com/risk-compliance-journal/th...
3,2026-11-16-12-56-00 +0000,wsj,Oil Prices Reverse Initial Gains as Supply Con...,https://www.wsj.com/finance/commodities-future...
4,2026-11-16-12-54-55 +0000,nyt,Bahrain Convicts Dozens of Shiite Clerics in S...,https://www.nytimes.com/2026/11/16/world/middl...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2753/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
175,trump,23
121,new,21
116,nvidia,16
86,he,12
17,ground,11


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
234,2026-11-15-18-18-45 +0000,nypost,Bessent urges Nvidia to look outside the compa...,https://nypost.com/2026/11/15/us-news/treasury...,73
60,2026-11-16-11-00-00 +0000,nypost,President Trump wishes happy 225th anniversary...,https://nypost.com/2026/11/16/us-news/presiden...,73
248,2026-11-15-17-27-08 +0000,cbc,Trump breaks silence on FAA's nationwide groun...,https://www.cbc.ca/news/world/trump-faa-ground...,67
251,2026-11-15-17-10-54 +0000,nypost,Trump blames Biden admin for FAA outage debacl...,https://nypost.com/2026/11/15/us-news/trump-bl...,63
103,2026-11-16-10-00-00 +0000,wapo,"Midterm losses, Nvidia scandal weaken Trump as...",https://www.washingtonpost.com/politics/2026/1...,63


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
234,73,2026-11-15-18-18-45 +0000,nypost,Bessent urges Nvidia to look outside the compa...,https://nypost.com/2026/11/15/us-news/treasury...
248,67,2026-11-15-17-27-08 +0000,cbc,Trump breaks silence on FAA's nationwide groun...,https://www.cbc.ca/news/world/trump-faa-ground...
24,53,2026-11-16-12-04-00 +0000,wsj,A Texas-based U.S. attorney sent what he calle...,https://www.wsj.com/arts-culture/books/the-whi...
175,39,2026-11-15-23-35-02 +0000,cbc,'I cheated': South Korean admits he used AI to...,https://www.cbc.ca/news/world/south-korea-tax-...
62,34,2026-11-16-11-00-00 +0000,nypost,NYC uses twice as many delivery contractors as...,https://nypost.com/2026/11/16/business/nyc-use...
223,33,2026-11-15-19-30-51 +0000,chitri,Israel kills a top Hamas commander as Jordan s...,https://www.chicagotribune.com/2026/11/15/isra...
190,33,2026-11-15-22-13-00 +0000,wsj,"Miguel Ángel Félix Gallardo, Architect of Mexi...",https://www.wsj.com/world/americas/miguel-ange...
98,30,2026-11-16-10-00-37 +0000,wsj,Stock Market Today: Tech Futures Slump as Nvid...,https://www.wsj.com/livecoverage/stock-market-...
60,29,2026-11-16-11-00-00 +0000,nypost,President Trump wishes happy 225th anniversary...,https://nypost.com/2026/11/16/us-news/presiden...
23,28,2026-11-16-12-05-27 +0000,nypost,Pentagon ignored warnings before Iran drone ki...,https://nypost.com/2026/11/16/us-news/pentagon...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
